# Exploration

Open-ended EDA and results review — distributions, skew, engineered features, and (once `src/train.py` has been run) how the LSTM compares to the naive/ARIMA/SARIMAX baselines. Nothing in this notebook feeds back into the pipeline; it's for understanding, not for producing files `train.py`/`inference.py` depend on.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import config
from create_folds import load_raw, add_calendar_features

df = load_raw()
df_model = add_calendar_features(df)
df_model.head()

## Distribution & skewness

Checks whether Sales and related raw columns are skewed enough to warrant a transform before modelling.

In [ ]:
cols = ['Sales', 'Cost Of Sales', 'Quantity Sold', 'Gross Profit']
cols = [c for c in cols if c in df.columns]
print('Skewness:')
print(df[cols].skew())

for col in cols:
    fig = px.histogram(df, x=col, nbins=30, marginal='box',
                        title=f'{col} Distribution | Skewness = {df[col].skew():.2f}')
    fig.update_layout(template='plotly_white', height=450)
    fig.show()

## Engineered calendar features

Sanity-check the cyclical encodings that `create_folds.py` produces — these are the only features the model sees besides Sales itself.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_model['Date'], y=df_model['dow_sin'], mode='lines', name='dow_sin'))
fig.add_trace(go.Scatter(x=df_model['Date'], y=df_model['doy_sin'], mode='lines', name='doy_sin'))
fig.update_layout(title='Cyclical Calendar Features', template='plotly_white', height=400)
fig.show()

## Model comparison (after running `src/train.py`)

Load the printed/saved metrics and compare LSTM against the naive, seasonal-naive, ARIMA, and SARIMAX baselines.

In [ ]:
# Paste in or re-load the comparison_metrics DataFrame produced by src/train.py
# comparison_metrics = pd.read_csv('../output/comparison_metrics.csv')
# fig = px.bar(comparison_metrics, x='Model', y='MAE', title='MAE by Model')
# fig.show()

## 2017 forecast review (after running `src/inference.py`)

In [ ]:
daily = pd.read_csv(config.DAILY_FORECAST_FILE, parse_dates=['Date'])
monthly = pd.read_csv(config.MONTHLY_FORECAST_FILE)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_model['Date'], y=df_model['Sales'], mode='lines', name='Historical Sales'))
fig.add_trace(go.Scatter(x=daily['Date'], y=daily['Predicted Sales'], mode='lines', name='2017 LSTM Forecast'))
fig.update_layout(title='Historical Sales and 2017 LSTM Forecast', template='plotly_white', height=600)
fig.show()

monthly